# 02c RQ2 exploratory: post-snapshot volume collider check

**Exploratory (Section 8).** Nothing here is pre-registered, and none of it changes the
confirmatory RQ2 result in notebook 02.

**Why.** Section 3 excludes lifetime `volume_usd` from the H2b liquidity covariate because
it includes trades after the snapshot horizon. However, the primary population filter
(`volume_usd >= $1,000`) and sensitivity 5.2 (`volume_usd >= $10,000`) still condition on
lifetime `volume_usd`. For a 1-day longshot (< 10¢), post-snapshot trading volume is
strongly driven by whether the longshot stages a comeback and wins. Conditioning on
`pre_snapshot_liquidity + post_snapshot_volume >= floor` therefore induces collider bias:
markets with low pre-snapshot liquidity can only enter the sample if they attract heavy
post-snapshot volume, which selects for winners.

This notebook tests that mechanism directly:
1. Measure how strongly post-snapshot volume (`volume_usd - liquidity_usd`) separates
   winning from losing longshots.
2. Re-estimate H2a and H2b using **pre-snapshot** `liquidity_usd` floors (`>= $1,000` and
   `>= $10,000`) instead of lifetime `volume_usd` floors.
3. Compare liquidity quintiles with and without lifetime-volume selection.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data" / "processed"
RESULTS = ROOT / "results"
LONGSHOT = 0.10
pd.set_option("display.width", 200)

cols = ["market_id", "horizon", "snap_price", "within_cap", "volume_usd",
        "liquidity_usd", "format", "subject", "cluster_id", "token1_won"]
frame = pd.read_parquet(DATA / "mart_analysis.parquet", columns=cols)
d1 = frame[(frame["horizon"] == "1d") & frame["within_cap"]].copy()
p = np.round(d1["snap_price"], 6)
t1 = p < 0.5
d1["ls_price"] = np.where(t1, p, np.round(1 - p, 6))
d1["won"] = np.where(t1, d1["token1_won"], ~d1["token1_won"]).astype(float)
d1["log_liq"] = np.log1p(d1["liquidity_usd"])
d1["template"] = (d1["format"] == "template").astype(float)
d1["post_vol"] = np.maximum(d1["volume_usd"] - d1["liquidity_usd"], 0.0)
ls_all = d1[d1["ls_price"] < LONGSHOT].reset_index(drop=True)
len(ls_all)


170618

## E1. Does post-snapshot volume select for winning longshots?

Among all 1-day within-cap longshots with **low pre-snapshot liquidity** (`liquidity_usd < $1,000`
or `< $10,000`), compare how often winning vs. losing longshots cross the lifetime `volume_usd`
floor after the snapshot.


In [2]:
mech_rows = []
for floor in (1_000, 10_000):
    sub = ls_all[ls_all["liquidity_usd"] < floor]
    for won_val, g in sub.groupby("won"):
        mech_rows.append({
            "floor_usd": floor,
            "won": int(won_val),
            "n_below_pre_floor": len(g),
            "share_crossing_lifetime_floor": (g["volume_usd"] >= floor).mean(),
            "median_post_vol_usd": g["post_vol"].median(),
            "mean_post_vol_usd": g["post_vol"].mean(),
        })
mech = pd.DataFrame(mech_rows)
mech.to_csv(RESULTS / "rq2_collider_mechanism.csv", index=False)
mech.round(4)


,floor_usd,won,n_below_pre_floor,share_crossing_lifetime_floor,median_post_vol_usd,mean_post_vol_usd
0,1000,0,109118,0.7379,2350.0357,7751.3496
1,1000,1,2378,0.7557,4814.7683,20392.8452
2,10000,0,142094,0.2752,3486.7542,15459.4100
3,10000,1,3079,0.4411,7196.7235,33423.2194


## E2. H2a and H2b under lifetime `volume_usd` vs. pre-snapshot `liquidity_usd` floors

Same CR1 cluster-robust OLS specifications as notebook 02 (`H2a`: intercept-only on
`won - ls_price`; `H2b`: `log1p(liquidity_usd)` + format FE). Coefficients and 95% CIs only.


In [3]:
def fit_h2(d):
    y = (d["won"] - d["ls_price"]).to_numpy()
    g = pd.factorize(d["cluster_id"])[0]
    r_a = sm.OLS(y, np.ones(len(d))).fit(cov_type="cluster", cov_kwds={"groups": g})
    X_cols = [np.ones(len(d)), d["log_liq"].to_numpy()]
    if d["template"].nunique() > 1:
        X_cols.append(d["template"].to_numpy())
    r_b = sm.OLS(y, np.column_stack(X_cols)).fit(cov_type="cluster", cov_kwds={"groups": g})
    return {
        "n": len(d),
        "clusters": int(g.max() + 1),
        "share_event": (d["format"] == "event").mean(),
        "h2a": r_a.params[0],
        "h2a_lo": r_a.params[0] - 1.959964 * r_a.bse[0],
        "h2a_hi": r_a.params[0] + 1.959964 * r_a.bse[0],
        "h2b": r_b.params[1],
        "h2b_lo": r_b.params[1] - 1.959964 * r_b.bse[1],
        "h2b_hi": r_b.params[1] + 1.959964 * r_b.bse[1],
    }

specs = {
    "no floor (5.1)": ls_all,
    "lifetime volume >= $1k (primary)": ls_all[ls_all["volume_usd"] >= 1_000],
    "pre-snapshot liq >= $1k": ls_all[ls_all["liquidity_usd"] >= 1_000],
    "lifetime volume >= $10k (5.2)": ls_all[ls_all["volume_usd"] >= 10_000],
    "pre-snapshot liq >= $10k": ls_all[ls_all["liquidity_usd"] >= 10_000],
}
floors = pd.DataFrame([{"sample": name, **fit_h2(df)} for name, df in specs.items()])
floors.to_csv(RESULTS / "rq2_collider_floors.csv", index=False)
floors.round(5)


,sample,n,clusters,share_event,h2a,h2a_lo,h2a_hi,h2b,h2b_lo,h2b_hi
0,no floor (5.1),170618,11378,0.35626,-0.00267,-0.00383,-0.00150,-0.00013,-0.00055,0.00029
1,lifetime volume >= $1k (primary),141440,10870,0.34318,-0.00232,-0.00338,-0.00125,-0.00034,-0.00072,0.00004
2,pre-snapshot liq >= $1k,59122,8959,0.47788,-0.00162,-0.00301,-0.00022,0.00024,-0.00044,0.00092
3,lifetime volume >= $10k (5.2),65908,8515,0.46296,0.00433,0.00201,0.00665,-0.00244,-0.00337,-0.00152
4,pre-snapshot liq >= $10k,25445,5806,0.58809,-0.00185,-0.00377,0.00007,0.00009,-0.00094,0.00112


## E3. Liquidity quintiles: lifetime floor vs. no floor vs. pre-snapshot floor

Check whether the non-monotone Q1 anomaly in the primary sample disappears once the
lifetime-volume selection filter is removed.


In [4]:
def quintile_table(df, label):
    q = df.assign(liq_quintile=pd.qcut(df["log_liq"], 5, labels=False, duplicates="drop") + 1)
    rows = []
    for k, g in q.groupby("liq_quintile"):
        y = (g["won"] - g["ls_price"]).to_numpy()
        cl = pd.factorize(g["cluster_id"])[0]
        r = sm.OLS(y, np.ones(len(g))).fit(cov_type="cluster", cov_kwds={"groups": cl})
        b, se = r.params[0], r.bse[0]
        rows.append({
            "sample": label,
            "liq_quintile": int(k),
            "n": len(g),
            "clusters": int(cl.max() + 1),
            "median_liq_usd": g["liquidity_usd"].median(),
            "median_post_vol_usd": g["post_vol"].median(),
            "mean_resid": b,
            "ci_low": b - 1.959964 * se,
            "ci_high": b + 1.959964 * se,
        })
    return pd.DataFrame(rows)

quints = pd.concat([
    quintile_table(ls_all[ls_all["volume_usd"] >= 1_000], "lifetime volume >= $1k"),
    quintile_table(ls_all, "no floor (5.1)"),
    quintile_table(ls_all[ls_all["liquidity_usd"] >= 1_000], "pre-snapshot liq >= $1k"),
], ignore_index=True)
quints.to_csv(RESULTS / "rq2_collider_quintiles.csv", index=False)
quints.round(5)


,sample,liq_quintile,n,clusters,median_liq_usd,median_post_vol_usd,mean_resid,ci_low,ci_high
0,lifetime volume >= $1k,1,28288,1872,15.640,2872.02096,0.00045,-0.00204,0.00293
1,lifetime volume >= $1k,2,28289,2036,178.910,3683.82792,-0.00482,-0.00652,-0.00312
2,lifetime volume >= $1k,3,28287,2631,603.390,5433.72018,-0.00419,-0.00613,-0.00224
3,lifetime volume >= $1k,4,28289,4316,2546.130,12438.43315,-0.00151,-0.00346,0.00044
4,lifetime volume >= $1k,5,28287,6258,31624.700,106424.07119,-0.00151,-0.00337,0.00036
5,no floor (5.1),1,34125,1841,2.420,892.90267,-0.00083,-0.00428,0.00263
6,no floor (5.1),2,34123,2224,75.860,2139.95159,-0.00352,-0.00526,-0.00178
7,no floor (5.1),3,34123,2553,338.660,4026.55042,-0.00458,-0.00627,-0.00289
8,no floor (5.1),4,34123,4182,1477.920,8869.62803,-0.00281,-0.00445,-0.00117
9,no floor (5.1),5,34124,7013,22744.805,80846.33149,-0.00160,-0.00332,0.00012


## E4. Event vs. template split under pre-snapshot liquidity floors


In [5]:
fmt_rows = []
for name in ("no floor (5.1)", "lifetime volume >= $1k (primary)", "pre-snapshot liq >= $1k",
             "lifetime volume >= $10k (5.2)", "pre-snapshot liq >= $10k"):
    df = specs[name]
    for fmt, g in df.groupby("format"):
        y = (g["won"] - g["ls_price"]).to_numpy()
        cl = pd.factorize(g["cluster_id"])[0]
        r = sm.OLS(y, np.ones(len(g))).fit(cov_type="cluster", cov_kwds={"groups": cl})
        b, se = r.params[0], r.bse[0]
        fmt_rows.append({
            "sample": name, "format": fmt, "n": len(g), "clusters": int(cl.max() + 1),
            "mean_resid": b, "ci_low": b - 1.959964 * se, "ci_high": b + 1.959964 * se,
        })
fmt_df = pd.DataFrame(fmt_rows)
fmt_df.round(5)


,sample,format,n,clusters,mean_resid,ci_low,ci_high
0,no floor (5.1),event,60784,10854,-0.00041,-0.00318,0.00237
1,no floor (5.1),template,109834,549,-0.00392,-0.00486,-0.00298
2,lifetime volume >= $1k (primary),event,48539,10373,0.00005,-0.00224,0.00235
3,lifetime volume >= $1k (primary),template,92901,522,-0.00355,-0.00460,-0.00250
4,pre-snapshot liq >= $1k,event,28253,8554,-0.00313,-0.00495,-0.00131
5,pre-snapshot liq >= $1k,template,30869,422,-0.00024,-0.00251,0.00204
6,lifetime volume >= $10k (5.2),event,30513,8144,0.00061,-0.00164,0.00286
7,lifetime volume >= $10k (5.2),template,35395,387,0.00753,0.00310,0.01197
8,pre-snapshot liq >= $10k,event,14964,5555,-0.00377,-0.00575,-0.00178
9,pre-snapshot liq >= $10k,template,10481,256,0.00089,-0.00358,0.00535


## Summary (exploratory)

- **Post-snapshot volume strongly selects for winning longshots (E1).** Among 1-day
  longshots with under $10,000 of pre-snapshot liquidity, 44.1% of winners cross the
  $10,000 lifetime-volume floor afterward compared with 27.5% of losers (median
  post-snapshot volume $7,197 vs. $3,487).
- **Replacing lifetime `volume_usd` with pre-snapshot `liquidity_usd` removes the 5.2
  sign flip (E2).** Under pre-snapshot `liquidity_usd >= $1,000`, H2a stays negative
  (−0.00162 [−0.00301, −0.00022]) and H2b turns slightly positive (+0.00024, CI covers 0).
  Under pre-snapshot `liquidity_usd >= $10,000`, H2a stays negative (−0.00185
  [−0.00377, +0.00007]) rather than flipping to +0.00433, and H2b is a clean null
  (+0.00009 [−0.00094, +0.00112]) rather than −0.00244.
- **Quintile 1's anomaly tracks the lifetime floor (E3).** Under the pre-snapshot
  $1,000 floor, the thinnest quintile is the most overpriced (−0.39¢ [−0.68, −0.10]).
- **Reading:** H2a's overpricing is more robust to deeper markets than sensitivity 5.2
  made it look, while H2b remains a clean null once post-snapshot volume selection is
  removed.
